
# 練習問題: べき乗法で固有振動モードを求める

## 目標

膜 (太鼓の面) や弦の **基本振動モード (一番ゆっくり揺れる定在波の形)** を, 反復解法の定番 **べき乗法 (power iteration)** で求める。べき乗法の中身は **行列ベクトル積 (matvec)** と **内積 (reduction)** の繰り返しで, どれもこれまで学んだ並列化がそのまま使える。

ここで使う行列 `A` は **B1 (2D熱伝導) や G1 (CG) と全く同じ 2次元ラプラシアン行列**。同じ行列を, B1 では「定常状態を反復で求め」, G1 では「方程式 `A x = b` として解いた」。ここ G2 では **同じ行列の固有振動モード (定在波の形) を出す**。1次元なら基本モードは `sin` の半波長, 2次元 (この問題) では中央が一番大きく盛り上がる滑らかな1つの山になる。

## べき乗法と「シフト」のしくみ

ある行列を, あるベクトルに何度も掛け続けると, ベクトルの向きはその行列の **最大固有値** に対応する固有ベクトルへ収束する (大きい固有値の成分だけが相対的に伸びるため)。これがべき乗法。

ところが今欲しいのは **最小** 固有値 `lambda_min` (= 一番ゆっくりの基本振動モード)。そこで **シフト** の工夫をする。`sigma = 8.0` (`A` の最大固有値 ≈ 8 より大きい) として

```
B = sigma * I - A
```

を考えると, `B` の最大固有値は `sigma - lambda_min(A)` になり, その固有ベクトルは `A` の最小固有モード (= 基本振動モード) と一致する。`B` にべき乗法を掛ければ目的が達成できる。

## 課題

`vibration.cpp` (または `vibration.f90`) は, 行列フリーのラプラシアン `A` に対し, 初期ベクトルを 1 から始めて `B = sigma*I - A` のべき乗法を回す。各反復で

```
Ax = A x
y  = sigma * x - Ax          (= B x)
lamB = dot(x, y) / dot(x, x)  (レイリー商)
x  = y / ||y||                (正規化)
```

を行い, `lamB` の変化が `tol` 未満になったら収束。最後に `lambda_min = sigma - lamB`。

並列化すべき2つのカーネルが `TODO` になっている (G1 CG と全く同じ2つ):

1. **matvec** (`y = A p`): 格子点の二重ループ。
   - C++: `#pragma omp parallel for collapse(2)`
   - Fortran: `!$omp parallel do collapse(2) private(v)` … `!$omp end parallel do`
2. **dot** (内積 `a・b`): 総和。
   - C++: `#pragma omp parallel for reduction(+:s)`
   - Fortran: `!$omp parallel do reduction(+:s)` … `!$omp end parallel do`

(`y = sigma*x - Ax` と正規化のループは逐次のまま書いてある。余力があればそこも並列化してよい。)

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore vibration.cpp -o vibration.exe

# Fortran
nvfortran -fast -mp=multicore vibration.f90 -o vibration.exe
```

第1引数で格子の一辺 `n`, 第2引数で収束しきい値 `tol`, 第3引数で最大反復回数。

```
OMP_NUM_THREADS=4 ./vibration.exe 128 1e-9 100000
```

## 期待される結果

```
n=128, iters=..., lambda_min=0.0005955653, analytic=0.0005955653, rel.err=...e-...
固有ベクトル(基本振動モード) 相対L2誤差=...e-..., 中央値=0.00... vs 隅の値=0.00...
```

- 計算した `lambda_min` が解析解 `4 - 4 cos(pi/(n+1))` とよく一致する (相対誤差が小さい)。
- 固有ベクトルは解析解 `sin(pi(i+1)/(n+1)) sin(pi(j+1)/(n+1))` (中央が最大の滑らかな1つの山) に一致し, 相対 L2 誤差が小さい。**中央値が最大, 隅はほぼ 0** なのが基本振動モードの形。
- べき乗法は収束が遅いことがあるので反復回数は多め (数千〜)。遅すぎる場合は `n=96` で試すとよい。
- `n` を大きくすると1反復の matvec と内積が重くなり, 並列化の効果 (台数効果) が見えやすい。「性能を比べる」セルで測ってみよ。
- (発展) 同じ matvec を GPU にオフロードする, `y = sigma*x - Ax` や正規化も並列化する, などで更に速くできる。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ vibration.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

/* 状態を持たない乱数 (初期ベクトルの生成用): (seed,k) から [0,1)。 */
static inline double draw_rand01(long long seed, long long k) {
  const long long M = 2147483647LL;
  long long x = ((seed % M) * 2654435761LL + (k % M) + 1) % M;
  x = ((x ^ (x >> 16)) * 1812433253LL) % M;
  x = ((x ^ (x >> 13)) * 1664525LL)    % M;
  x =  (x ^ (x >> 16)) % M;
  return (double)x / (double)M;
}

/* 行列ベクトル積 y = A p。
   A は n×n 格子上の 2次元ラプラシアン (5点ステンシル, ディリクレ境界=0)。
   (A p)[i,j] = 4 p[i,j] - p[i-1,j] - p[i+1,j] - p[i,j-1] - p[i,j+1]  (領域外は 0)。
   これは B1 (2D熱伝導) や G1 (CG) と同じラプラシアン行列である。 */
void matvec(int n, const double * p, double * y) {
  // TODO: 格子点の二重ループを #pragma omp parallel for collapse(2) で並列化せよ.
  for (int i = 0; i < n; i++) {
    for (int j = 0; j < n; j++) {
      double v = 4.0 * p[i*n+j];
      if (i > 0)     v -= p[(i-1)*n+j];
      if (i < n - 1) v -= p[(i+1)*n+j];
      if (j > 0)     v -= p[i*n+j-1];
      if (j < n - 1) v -= p[i*n+j+1];
      y[i*n+j] = v;
    }
  }
}

/* 内積 a・b */
double dot(long N, const double * a, const double * b) {
  double s = 0.0;
  // TODO: 内積の和を #pragma omp parallel for reduction(+:s) で並列化せよ.
  for (long k = 0; k < N; k++) s += a[k] * b[k];
  return s;
}

int main(int argc, char ** argv) {
  int    n     = (argc > 1 ? atoi(argv[1]) : 128);   /* 格子の一辺 (未知数は N = n*n) */
  double tol   = (argc > 2 ? atof(argv[2]) : 1e-9);
  int    maxit = (argc > 3 ? atoi(argv[3]) : 100000);
  long   N     = (long)n * n;
  double sigma = 8.0;                                /* シフト量 (> lambda_max ≈ 8) */

  double * x  = (double *)malloc(sizeof(double) * N);
  double * y  = (double *)malloc(sizeof(double) * N);
  double * Ax = (double *)malloc(sizeof(double) * N);

  /* 初期ベクトル: すべて 1 から始めて正規化 */
  for (long k = 0; k < N; k++) x[k] = 1.0;
  double nrm0 = sqrt(dot(N, x, x));
  for (long k = 0; k < N; k++) x[k] /= nrm0;

  /* べき乗法: B = sigma*I - A を繰り返し掛ける。
     B の最大固有値 = sigma - lambda_min(A) に収束し, 固有ベクトルは基本振動モード。 */
  int it;
  double lamB = 0.0, lamB_prev = 0.0;
  double t0 = omp_get_wtime();
  for (it = 0; it < maxit; it++) {
    matvec(n, x, Ax);
    for (long k = 0; k < N; k++) y[k] = sigma * x[k] - Ax[k];   /* y = B x (逐次のまま) */
    lamB = dot(N, x, y) / dot(N, x, x);                          /* レイリー商 */
    double nrm = sqrt(dot(N, y, y));
    for (long k = 0; k < N; k++) x[k] = y[k] / nrm;              /* 正規化 (逐次のまま) */
    if (it > 0 && fabs(lamB - lamB_prev) < tol) { it++; break; }
    lamB_prev = lamB;
  }
  double elapsed = omp_get_wtime() - t0;

  double lambda_min = sigma - lamB;
  double analytic   = 4.0 - 4.0 * cos(M_PI / (n + 1));
  double rel_err    = fabs(lambda_min - analytic) / analytic;

  /* 固有ベクトルの検算: 解析解 v[i,j] = sin(pi(i+1)/(n+1)) sin(pi(j+1)/(n+1)) と比べる。
     まず符号を合わせ, 正規化した両者の相対 L2 誤差を計算する。 */
  double * ve = (double *)malloc(sizeof(double) * N);
  double vn = 0.0;
  for (int i = 0; i < n; i++)
    for (int j = 0; j < n; j++) {
      double v = sin(M_PI * (i+1) / (n+1)) * sin(M_PI * (j+1) / (n+1));
      ve[i*n+j] = v; vn += v * v;
    }
  vn = sqrt(vn);
  double sgn = (x[(n/2)*n + n/2] >= 0.0) ? 1.0 : -1.0;   /* 中央の値で符号を合わせる */
  double vecerr = 0.0;
  for (long k = 0; k < N; k++) {
    double d = sgn * x[k] - ve[k] / vn;
    vecerr += d * d;
  }
  vecerr = sqrt(vecerr);   /* x も ve/vn も単位ベクトルなので, これが相対 L2 誤差 */

  printf("n=%d, iters=%d, lambda_min=%.10f, analytic=%.10f, rel.err=%.2e\n",
         n, it, lambda_min, analytic, rel_err);
  printf("固有ベクトル(基本振動モード) 相対L2誤差=%.2e, 中央値=%.4f vs 隅の値=%.4f\n",
         vecerr, sgn * x[(n/2)*n + n/2], sgn * x[0]);
  printf("elapsed = %.3f sec\n", elapsed);
  free(x); free(y); free(Ax); free(ve);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore vibration.cpp -o vibration_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./vibration_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ vibration.f90
module vibration_mod
contains
  ! 状態を持たない乱数 (初期ベクトルの生成用): (seed,k) から [0,1)。
  function draw_rand01(seed, k) result(u)
    integer(8), intent(in) :: seed, k
    real(8) :: u
    integer(8), parameter :: M = 2147483647_8
    integer(8) :: x
    x = modulo(modulo(seed, M) * 2654435761_8 + modulo(k, M) + 1_8, M)
    x = modulo(ieor(x, ishft(x, -16)) * 1812433253_8, M)
    x = modulo(ieor(x, ishft(x, -13)) * 1664525_8,    M)
    x = modulo(ieor(x, ishft(x, -16)), M)
    u = real(x, 8) / real(M, 8)
  end function draw_rand01

  ! 行列ベクトル積 y = A p。A は n×n 格子の 2次元ラプラシアン (5点ステンシル,
  ! ディリクレ境界=0)。これは B1 (2D熱伝導) や G1 (CG) と同じラプラシアン行列。
  ! 添字は 0..n*n-1 (idx = i*n + j)。
  subroutine matvec(n, p, y)
    integer, intent(in) :: n
    real(8), intent(in) :: p(0:n*n-1)
    real(8), intent(out) :: y(0:n*n-1)
    integer :: i, j
    real(8) :: v
    ! TODO: 格子点の二重ループを !$omp parallel do collapse(2) private(v) で並列化せよ.
    do i = 0, n - 1
       do j = 0, n - 1
          v = 4.0d0 * p(i*n+j)
          if (i > 0)     v = v - p((i-1)*n+j)
          if (i < n - 1) v = v - p((i+1)*n+j)
          if (j > 0)     v = v - p(i*n+j-1)
          if (j < n - 1) v = v - p(i*n+j+1)
          y(i*n+j) = v
       end do
    end do
    ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do)。
  end subroutine matvec

  ! 内積 a・b
  function dot(N, a, b) result(s)
    integer(8), intent(in) :: N
    real(8), intent(in) :: a(0:N-1), b(0:N-1)
    real(8) :: s
    integer(8) :: k
    s = 0.0d0
    ! TODO: 内積の和を !$omp parallel do reduction(+:s) で並列化せよ.
    do k = 0, N - 1
       s = s + a(k) * b(k)
    end do
    ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do)。
  end function dot
end module vibration_mod

program vibration
  use vibration_mod
  use omp_lib
  character(len=32) :: arg
  integer :: n, maxit, it, i, j
  real(8) :: tol, sigma, lamB, lamB_prev, nrm, nrm0, lambda_min, analytic, rel_err
  real(8) :: vn, sgn, vecerr, d, v, pi
  integer(8) :: N8, k
  real(8), allocatable :: x(:), y(:), Ax(:), ve(:)
  pi = 4.0d0 * atan(1.0d0)
  n = 128; tol = 1d-9; maxit = 100000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) tol
  end if
  if (command_argument_count() >= 3) then
     call get_command_argument(3, arg); read (arg, *) maxit
  end if
  N8 = int(n, 8) * n
  sigma = 8.0d0                        ! シフト量 (> lambda_max ≈ 8)

  allocate(x(0:N8-1), y(0:N8-1), Ax(0:N8-1), ve(0:N8-1))

  ! 初期ベクトル: すべて 1 から始めて正規化
  do k = 0, N8 - 1
     x(k) = 1.0d0
  end do
  nrm0 = sqrt(dot(N8, x, x))
  do k = 0, N8 - 1
     x(k) = x(k) / nrm0
  end do

  ! べき乗法: B = sigma*I - A を繰り返し掛ける。
  ! B の最大固有値 = sigma - lambda_min(A) に収束し, 固有ベクトルは基本振動モード。
  lamB = 0.0d0; lamB_prev = 0.0d0
  t0_block: block
    real(8) :: t0, elapsed
    t0 = omp_get_wtime()
    do it = 1, maxit
       call matvec(n, x, Ax)
       do k = 0, N8 - 1
          y(k) = sigma * x(k) - Ax(k)     ! y = B x (逐次のまま)
       end do
       lamB = dot(N8, x, y) / dot(N8, x, x)   ! レイリー商
       nrm = sqrt(dot(N8, y, y))
       do k = 0, N8 - 1
          x(k) = y(k) / nrm                ! 正規化 (逐次のまま)
       end do
       if (it > 1 .and. abs(lamB - lamB_prev) < tol) exit
       lamB_prev = lamB
    end do
    elapsed = omp_get_wtime() - t0

    lambda_min = sigma - lamB
    analytic   = 4.0d0 - 4.0d0 * cos(pi / (n + 1))
    rel_err    = abs(lambda_min - analytic) / analytic

    ! 固有ベクトルの検算: 解析解 v[i,j] = sin(pi(i+1)/(n+1)) sin(pi(j+1)/(n+1))。
    vn = 0.0d0
    do i = 0, n - 1
       do j = 0, n - 1
          v = sin(pi * (i+1) / (n+1)) * sin(pi * (j+1) / (n+1))
          ve(i*n+j) = v; vn = vn + v * v
       end do
    end do
    vn = sqrt(vn)
    if (x((n/2)*n + n/2) >= 0.0d0) then
       sgn = 1.0d0
    else
       sgn = -1.0d0
    end if
    vecerr = 0.0d0
    do k = 0, N8 - 1
       d = sgn * x(k) - ve(k) / vn
       vecerr = vecerr + d * d
    end do
    vecerr = sqrt(vecerr)

    print "(a,i0,a,i0,a,f0.10,a,f0.10,a,es9.2)", &
         "n=", n, ", iters=", min(it, maxit), ", lambda_min=", lambda_min, &
         ", analytic=", analytic, ", rel.err=", rel_err
    print "(a,es9.2,a,f0.4,a,f0.4)", &
         "固有ベクトル(基本振動モード) 相対L2誤差=", vecerr, &
         ", 中央値=", sgn * x((n/2)*n + n/2), " vs 隅の値=", sgn * x(0)
    print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
  end block t0_block
end program vibration

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore vibration.f90 -o vibration_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./vibration_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:vibration.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:vibration.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:vibration.cpp}